# TABR datatest event inference — 20 selected subjects

Evaluate datatest 15/17/mixed on selected 20 subjects only: 10 anxiety_tinggi + 10 anxiety_rendah.


In [1]:
from __future__ import annotations

import gc
import json
import os
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import Markdown, display
from scipy.special import expit
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix

sns.set_theme(style='whitegrid')
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

TABR_ROOT = ROOT / 'third_party' / 'tabular-dl-tabr-official'
os.environ['PROJECT_DIR'] = str(TABR_ROOT)
if str(TABR_ROOT) not in sys.path:
    sys.path.append(str(TABR_ROOT))

import lib
from bin.tabr import Model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EXP_ROOT = TABR_ROOT / 'exp' / 'tabr'
DATA_ROOT = TABR_ROOT / 'data'
FINAL_THRESHOLD = 0.5
TOP_K = 3
EVENT_METHODS = ['top3_mean', 'max']
EVAL_ROOT = ROOT / 'output/apex/datatest/model_eval_20_subjects'
CM_ROOT = EVAL_ROOT / 'confusion_matrices'
EVAL_ROOT.mkdir(parents=True, exist_ok=True)
(CM_ROOT / 'event_level').mkdir(parents=True, exist_ok=True)
(CM_ROOT / 'subject_level').mkdir(parents=True, exist_ok=True)
print(f'Root: {ROOT}')
print(f'Device: {device}')


Root: /home/ryuko/Documents/Codes/Python/Skripsi/Convat-1st
Device: cuda


In [2]:
MODEL_SPECS = [
    {
        'label': 'baseline_event_model',
        'exp_name': 'convat_apex_anxiety',
        'mode': 'baseline',
    },
    {
        'label': 'qholdout_q123_q4_q5',
        'exp_name': 'convat_apex_anxiety_qholdout_q123_q4_q5',
        'mode': 'qholdout',
    },
    {
        'label': 'qwalk_q12_q3_q4',
        'exp_name': 'convat_apex_anxiety_qwalk_q12_q3_q4',
        'mode': 'qwalkforward',
    },
    {
        'label': 'qwalk_q123_q4_q5',
        'exp_name': 'convat_apex_anxiety_qwalk_q123_q4_q5',
        'mode': 'qwalkforward',
    },
    {
        'label': 'qholdout_q123_q4_q5_alis',
        'exp_name': 'convat_apex_anxiety_qholdout_q123_q4_q5_alis',
        'mode': 'qholdout',
    },
    {
        'label': 'qwalk_q12_q3_q4',
        'exp_name': 'convat_apex_anxiety_qwalk_q12_q3_q4_alis',
        'mode': 'qwalkforward',
    },
    {
        'label': 'qwalk_q123_q4_q5',
        'exp_name': 'convat_apex_anxiety_qwalk_q123_q4_q5_alis',
        'mode': 'qwalkforward',
    },
]
model_specs_df = pd.DataFrame(MODEL_SPECS)
display(model_specs_df)


,label,exp_name,mode
0,baseline_event_model,convat_apex_anxiety,baseline
1,qholdout_q123_q4_q5,convat_apex_anxiety_qholdout_q123_q4_q5,qholdout
2,qwalk_q12_q3_q4,convat_apex_anxiety_qwalk_q12_q3_q4,qwalkforward
3,qwalk_q123_q4_q5,convat_apex_anxiety_qwalk_q123_q4_q5,qwalkforward
4,qholdout_q123_q4_q5_alis,convat_apex_anxiety_qholdout_q123_q4_q5_alis,qholdout
5,qwalk_q12_q3_q4,convat_apex_anxiety_qwalk_q12_q3_q4_alis,qwalkforward
6,qwalk_q123_q4_q5,convat_apex_anxiety_qwalk_q123_q4_q5_alis,qwalkforward


In [ ]:
FEATURE_PATHS = {
    '15_only': [ROOT / 'output/apex/datatest/features/15_04_2026/poc_abs_test.xlsx'],
    '17_only': [ROOT / 'output/apex/datatest/features/17_04_2026/poc_abs_test.xlsx'],
    '15_17_mixed': [
        ROOT / 'output/apex/datatest/features/15_04_2026/poc_abs_test.xlsx',
        ROOT / 'output/apex/datatest/features/17_04_2026/poc_abs_test.xlsx',
    ],
}

SELECTED_HIGH = [
    'naufal_abid_aurizky',
    'hafif_nurrahmad',
    'bisma_adhiaksa',
    'tsabita_keizya_vanya_putri',
    'raihan_daffa_izzuddin',
    'andy',
    'dicky_darmawan',
    'fata_haidar_aly',
    'davi_aulia_maghfirah',
    'siti_mutmainah',
]

SELECTED_LOW = [
    'muhammad_daniel_assaqovi',
    'sileysa_faedatul_nuraini',
    'ilham_dharma_atmaja',
    'ahmad_falahi',
    'aryan_zuda_firdaus',
    'abelas_solihin',
    'dewi_anggraini_meita_sari',
    'farrel_andika_chandra',
    'amin_aziz_sudjud',
    'zacky_rio_orlando',
]

SELECTED_SUBJECTS = SELECTED_HIGH + SELECTED_LOW
EXPECTED_LABEL_BY_BASE = {
    **{name: 'anxiety_tinggi' for name in SELECTED_HIGH},
    **{name: 'anxiety_rendah' for name in SELECTED_LOW},
}

META_COLS = {
    'date', 'phase', 'condition', 'label', 'participant', 'participant_raw', 'question', 'question_no',
    'sample', 'clip', 'event_clip', 'event_no', 'clip_path', 'frame', 'score', 'annotation_sheet',
}

def label_to_int(label: str) -> int:
    return 1 if str(label) in {'anxiety_tinggi', 'anxiety', '1', 'tinggi'} else 0


def participant_base(value: str) -> str:
    return re.sub(r'_\d+$', '', str(value))


def filter_selected_subjects(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['participant_base'] = out['participant_raw'].map(participant_base)
    out = out[out['participant_base'].isin(SELECTED_SUBJECTS)].copy()
    out = out[out.apply(lambda row: row['label'] == EXPECTED_LABEL_BY_BASE[row['participant_base']], axis=1)].copy()
    return out


def load_scenario_features(paths: list[Path]) -> pd.DataFrame:
    dfs = []
    for path in paths:
        if not path.exists():
            raise FileNotFoundError(f'Feature file not found: {path}')
        df = pd.read_excel(path)
        if 'date' not in df.columns:
            df['date'] = path.parent.name
        dfs.append(df)
    out = pd.concat(dfs, ignore_index=True)
    out = filter_selected_subjects(out)
    out['true_label'] = out['label'].map(label_to_int).astype(int)
    out['event_key'] = (
        out['date'].astype(str) + '|' + out['participant_raw'].astype(str) + '|' + out['question'].astype(str) + '|' +
        out['sample'].astype(str) + '|' + out['event_clip'].astype(str) + '|' + out['event_no'].astype(str)
    )
    out['subject_key'] = out['date'].astype(str) + '|' + out['participant_base'].astype(str)
    return out

scenario_frames = {}
for scenario, paths in FEATURE_PATHS.items():
    df = load_scenario_features(paths)
    scenario_frames[scenario] = df
    subject_counts = df.drop_duplicates('subject_key')['label'].value_counts().to_dict()
    missing = sorted(set(SELECTED_SUBJECTS) - set(df['participant_base'].unique()))
    print(scenario, df.shape, 'events=', df['event_key'].nunique(), 'subjects=', df['subject_key'].nunique(), 'subject_labels=', subject_counts, 'frame_labels=', df['label'].value_counts().to_dict())
    if missing:
        print('missing:', missing)

display(pd.DataFrame({
    'participant_base': SELECTED_SUBJECTS,
    'target_label': [EXPECTED_LABEL_BY_BASE[x] for x in SELECTED_SUBJECTS],
}))


In [ ]:
def load_best_seed_report(exp_name: str) -> tuple[int, dict, Path]:
    eval_dir = EXP_ROOT / exp_name / '0-evaluation'
    rows = []
    for run_dir in sorted([x for x in eval_dir.iterdir() if x.is_dir() and x.name.isdigit()]):
        report_path = run_dir / 'report.json'
        if not report_path.exists():
            continue
        report = json.loads(report_path.read_text())
        rows.append((int(run_dir.name), report, run_dir))
    if not rows:
        raise FileNotFoundError(f'No report.json found for {exp_name}')
    return max(rows, key=lambda x: x[1]['metrics']['val']['score'])


def load_run_config(exp_name: str, best_seed: int, report: dict) -> dict:
    seed_toml = EXP_ROOT / exp_name / f'{best_seed}.toml'
    if seed_toml.exists():
        return lib.load_config(seed_toml)
    if 'config' in report:
        return report['config']
    tuning_toml = EXP_ROOT / exp_name / '0-tuning.toml'
    if tuning_toml.exists():
        return lib.load_config(tuning_toml)
    raise FileNotFoundError(f'No config found for {exp_name} seed {best_seed}')


def resolve_data_dir(data_path: str) -> Path:
    if data_path.startswith(':data/'):
        return DATA_ROOT / data_path.removeprefix(':data/')
    return Path(data_path)


def load_model_bundle(exp_name: str) -> dict:
    best_seed, report, best_run_dir = load_best_seed_report(exp_name)
    cfg = load_run_config(exp_name, best_seed, report)
    data_dir = resolve_data_dir(cfg['data']['path'])
    feature_cols = json.loads((data_dir / 'feature_cols.json').read_text())
    dataset = lib.build_dataset(**cfg['data']).to_torch(device)
    model = Model(
        n_num_features=dataset.n_num_features,
        n_bin_features=dataset.n_bin_features,
        cat_cardinalities=dataset.cat_cardinalities(),
        n_classes=dataset.n_classes(),
        **cfg['model'],
    ).to(device)
    checkpoint = lib.load_checkpoint(best_run_dir)
    model.load_state_dict(checkpoint['model'])
    model.eval()
    candidate_x = {key[2:]: dataset.data[key]['train'] for key in dataset.data if key.startswith('X_')}
    candidate_y = dataset.Y['train']
    return {
        'model': model,
        'dataset': dataset,
        'candidate_x': candidate_x,
        'candidate_y': candidate_y,
        'feature_cols': feature_cols,
        'context_size': cfg['context_size'],
        'best_seed': best_seed,
        'data_path': cfg['data']['path'],
    }


def release_model_bundle(bundle: dict) -> None:
    del bundle
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
@torch.inference_mode()
def predict_frame_probs(bundle: dict, df: pd.DataFrame, batch_size: int = 512) -> np.ndarray:
    feature_cols = bundle['feature_cols']
    missing = [c for c in feature_cols if c not in df.columns]
    if missing:
        raise KeyError(f'Missing feature cols: {missing[:20]}')
    x_np = df[feature_cols].to_numpy(dtype=np.float32)
    probs = []
    for start in range(0, len(x_np), batch_size):
        xb = torch.as_tensor(x_np[start:start + batch_size], device=device)
        for i in range(xb.shape[0]):
            x = {'num': xb[i:i+1]}
            logit = bundle['model'](
                x_=x,
                y=None,
                candidate_x_=bundle['candidate_x'],
                candidate_y=bundle['candidate_y'],
                context_size=bundle['context_size'],
                is_train=False,
            ).squeeze(-1).item()
            probs.append(float(expit(logit)))
    return np.asarray(probs, dtype=float)


def aggregate_probs(values, method: str, top_k: int = TOP_K) -> float:
    arr = np.asarray(list(values), dtype=float)
    if len(arr) == 0:
        return np.nan
    if method == 'max':
        return float(np.max(arr))
    if method == 'top3_mean':
        k = min(top_k, len(arr))
        return float(np.mean(np.sort(arr)[-k:]))
    raise ValueError(f'Unknown aggregation method: {method}')


In [ ]:
def make_event_predictions(frame_df: pd.DataFrame, method: str) -> pd.DataFrame:
    rows = []
    group_cols = ['event_key']
    meta_cols = ['date', 'participant_raw', 'participant', 'subject_key', 'label', 'true_label', 'question', 'sample', 'event_clip', 'event_no']
    for event_key, g in frame_df.groupby(group_cols, sort=False):
        first = g.iloc[0]
        prob = aggregate_probs(g['prob_anxiety_tinggi'], method)
        row = {col: first[col] for col in meta_cols if col in g.columns}
        row.update({
            'event_key': event_key if isinstance(event_key, str) else event_key[0],
            'aggregation': method,
            'agg_prob_anxiety_tinggi': prob,
            'pred_label': int(prob >= FINAL_THRESHOLD),
            'n_frames': len(g),
        })
        rows.append(row)
    return pd.DataFrame(rows)


def make_subject_predictions(event_df: pd.DataFrame, method: str) -> pd.DataFrame:
    rows = []
    meta_cols = ['date', 'participant_raw', 'participant', 'subject_key', 'label', 'true_label']
    for subject_key, g in event_df.groupby('subject_key', sort=False):
        first = g.iloc[0]
        prob = aggregate_probs(g['agg_prob_anxiety_tinggi'], method)
        row = {col: first[col] for col in meta_cols if col in g.columns}
        row.update({
            'subject_key': subject_key,
            'aggregation': method,
            'agg_prob_anxiety_tinggi': prob,
            'pred_label': int(prob >= FINAL_THRESHOLD),
            'n_events': len(g),
        })
        rows.append(row)
    return pd.DataFrame(rows)


def summarize_predictions(df: pd.DataFrame, level: str, scenario: str, spec: dict, method: str) -> dict:
    y_true = df['true_label'].astype(int).to_numpy()
    y_pred = df['pred_label'].astype(int).to_numpy()
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    report = classification_report(y_true, y_pred, labels=[0, 1], target_names=['anxiety_rendah', 'anxiety_tinggi'], output_dict=True, zero_division=0)
    return {
        'level': level,
        'scenario': scenario,
        'exp_name': spec['exp_name'],
        'label': spec['label'],
        'mode': spec['mode'],
        'aggregation': method,
        'n': len(df),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
        'rendah_precision': report['anxiety_rendah']['precision'],
        'rendah_recall': report['anxiety_rendah']['recall'],
        'rendah_f1': report['anxiety_rendah']['f1-score'],
        'tinggi_precision': report['anxiety_tinggi']['precision'],
        'tinggi_recall': report['anxiety_tinggi']['recall'],
        'tinggi_f1': report['anxiety_tinggi']['f1-score'],
        'macro_f1': report['macro avg']['f1-score'],
    }


def safe_name(value: str) -> str:
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(value))


def plot_confusion(df: pd.DataFrame, level: str, scenario: str, spec: dict, method: str) -> Path:
    y_true = df['true_label'].astype(int).to_numpy()
    y_pred = df['pred_label'].astype(int).to_numpy()
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    out_dir = CM_ROOT / level
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / f"{safe_name(scenario)}__{safe_name(spec['label'])}__{safe_name(spec['exp_name'])}__{method}.png"
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['rendah', 'tinggi'], yticklabels=['rendah', 'tinggi'], ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(f"{level} | {scenario}\\n{spec['label']} | {method}")
    plt.tight_layout()
    fig.savefig(path, dpi=160)
    plt.show()
    plt.close(fig)
    return path




In [ ]:
all_frame_pred_parts = []
event_pred_parts = []
subject_pred_parts = []
event_summary_rows = []
subject_summary_rows = []

for spec in MODEL_SPECS:
    display(Markdown(f"## Model: `{spec['exp_name']}`"))
    bundle = load_model_bundle(spec['exp_name'])
    print('best_seed=', bundle['best_seed'], 'features=', len(bundle['feature_cols']), 'data_path=', bundle['data_path'])

    try:
        for scenario, base_df in scenario_frames.items():
            work = base_df.copy()
            work['prob_anxiety_tinggi'] = predict_frame_probs(bundle, work)
            work['pred_label'] = (work['prob_anxiety_tinggi'] >= FINAL_THRESHOLD).astype(int)
            work['scenario'] = scenario
            work['exp_name'] = spec['exp_name']
            work['model_label'] = spec['label']
            all_frame_pred_parts.append(work[[
                'scenario', 'exp_name', 'model_label', 'date', 'participant_base', 'participant_raw', 'participant', 'subject_key',
                'event_key', 'label', 'true_label', 'question', 'sample', 'event_clip', 'event_no', 'frame',
                'prob_anxiety_tinggi', 'pred_label'
            ]])

            for method in EVENT_METHODS:
                event_df = make_event_predictions(work, method)
                event_df['scenario'] = scenario
                event_df['exp_name'] = spec['exp_name']
                event_df['model_label'] = spec['label']
                event_pred_parts.append(event_df)
                event_summary_rows.append(summarize_predictions(event_df, 'event_level', scenario, spec, method))
                plot_confusion(event_df, 'event_level', scenario, spec, method)

                subject_df = make_subject_predictions(event_df, method)
                subject_df['scenario'] = scenario
                subject_df['exp_name'] = spec['exp_name']
                subject_df['model_label'] = spec['label']
                subject_pred_parts.append(subject_df)
                subject_summary_rows.append(summarize_predictions(subject_df, 'subject_level', scenario, spec, method))
                plot_confusion(subject_df, 'subject_level', scenario, spec, method)
    finally:
        release_model_bundle(bundle)

event_summary_df = pd.DataFrame(event_summary_rows)
subject_summary_df = pd.DataFrame(subject_summary_rows)
display(Markdown('## Event-level summary'))
display(event_summary_df.sort_values(['scenario', 'aggregation', 'accuracy', 'balanced_accuracy'], ascending=[True, True, False, False]).reset_index(drop=True))
display(Markdown('## Subject-level summary'))
display(subject_summary_df.sort_values(['scenario', 'aggregation', 'accuracy', 'balanced_accuracy'], ascending=[True, True, False, False]).reset_index(drop=True))


In [ ]:
frame_predictions_df = pd.concat(all_frame_pred_parts, ignore_index=True) if all_frame_pred_parts else pd.DataFrame()
event_predictions_df = pd.concat(event_pred_parts, ignore_index=True) if event_pred_parts else pd.DataFrame()
subject_predictions_df = pd.concat(subject_pred_parts, ignore_index=True) if subject_pred_parts else pd.DataFrame()

with pd.ExcelWriter(EVAL_ROOT / 'datatest_20_subjects_predictions.xlsx') as writer:
    frame_predictions_df.to_excel(writer, sheet_name='frame_predictions', index=False)
    event_predictions_df.to_excel(writer, sheet_name='event_predictions', index=False)
    subject_predictions_df.to_excel(writer, sheet_name='subject_predictions', index=False)

event_summary_df.to_excel(EVAL_ROOT / 'datatest_20_subjects_event_level_summary.xlsx', index=False)
subject_summary_df.to_excel(EVAL_ROOT / 'datatest_20_subjects_subject_level_summary.xlsx', index=False)

print(f'Saved: {EVAL_ROOT / "datatest_20_subjects_predictions.xlsx"}')
print(f'Saved: {EVAL_ROOT / "datatest_20_subjects_event_level_summary.xlsx"}')
print(f'Saved: {EVAL_ROOT / "datatest_20_subjects_subject_level_summary.xlsx"}')


In [ ]:
display(Markdown('## Best per scenario/method — event level'))
best_event = (
    event_summary_df
    .sort_values(['scenario', 'aggregation', 'accuracy', 'balanced_accuracy', 'macro_f1'], ascending=[True, True, False, False, False])
    .groupby(['scenario', 'aggregation'], as_index=False)
    .first()
)
display(best_event)

display(Markdown('## Best per scenario/method — subject level'))
best_subject = (
    subject_summary_df
    .sort_values(['scenario', 'aggregation', 'accuracy', 'balanced_accuracy', 'macro_f1'], ascending=[True, True, False, False, False])
    .groupby(['scenario', 'aggregation'], as_index=False)
    .first()
)
display(best_subject)
